# Task 0 — Baseline (원본 train만 사용)

**목적**: 원본 train 232장(corrections.json 반영)만으로 5-fold 학습하여, 합성 데이터 보강 실험(Task 1/2)의 비교 기준을 만든다.

| 항목 | 내용 |
|---|---|
| 학습 데이터 | 원본 train 232장 (annotation 수정 반영) |
| 클래스 | 56종 (라벨 1~56) |
| 모델/학습 | RF-DETR **medium**, 100 epochs, early stopping — Task 0/1/2 공통 고정 |
| 분할 | StratifiedGroupKFold 5-fold (희소 클래스 층화, 구성 코드 그룹핑) |
| test 추론 | 5-fold 앙상블 → **WBF** 융합 → score ≥ 0.3 제출 |
| 제출 파일 | `submission_task0_medium_task0_base.csv` |

**파이프라인 (셀 순서)**
1. 패키지 설치 → 2. 팀 저장소 clone + 함수 import → 3. Kaggle Input 경로 자동 탐색 → 4. 실험 설정(config)
5. 원본 train 로드 + corrections 적용 → 6. 데이터 병합: (생략 — Baseline은 원본만 사용, 자리만 유지)
7. 병합 데이터 클래스별 bbox 시각화 → 8. 카테고리 라벨 매핑 → 9. 5-fold 분할 + fold별 클래스 배분 점검
10. fold 디렉토리 생성 → 11. 5-fold 학습(fold별 학습곡선/오답 시각화 자동) → 12. 결과 요약(fold 평균·클래스별 mAP)
13. test 5-fold 앙상블 추론 → 14. WBF 융합 → 15. 클래스별 예측 시각화(confidence 표시) → 16. 제출 CSV 생성

**실행 전제**
- Kaggle Notebook: **GPU on / Internet on**
- Input 연결: competition 데이터(`train_images`, `train_annotations`, `test_images`)
- ⚠ medium × 100ep × 5fold는 세션 한도(12h)를 넘길 수 있음. `train_fold()`는 `backup_dir`에 체크포인트가 이미 있으면 그 fold를 건너뛰므로, 이전 커밋 output을 Input으로 추가해 복사하면 **이어하기** 가능 (11번 앞 선택 셀 참고).

**주의사항**
- 이 파일은 corrections 하드코딩 이전에 load하였던 파일로, **corrections 적용 로직은 task2, 3, 4를 따르는 것을 권장**합니다.

In [ ]:
# [1. 환경 준비] 필요 패키지 설치
#  - rfdetr[train]  : RF-DETR 학습/추론 (metrics.csv 로깅 포함)
#  - ensemble-boxes : 5-fold 예측 WBF(Weighted Box Fusion) 융합용
#  - torchmetrics   : mAP 계산 (저장소 visualize.py가 사용)
!pip install -q "rfdetr[train]" ensemble-boxes torchmetrics

In [ ]:
# [2. 저장소 준비] 팀 저장소 main 브랜치를 clone하고 RF_DETR_split_ver를 import 경로에 추가합니다.
#  ⚠ 저장소 파일/함수는 수정하지 않고 그대로 재사용합니다. (수정이 필요한 로직은 노트북 내 로컬 함수로 재정의)
import os, sys, re, glob, json, shutil
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image

REPO_URL = 'https://github.com/kim-tae-yoon-0718/ai12-team01.git'
REPO_DIR = '/kaggle/working/ai12-team01'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
sys.path.insert(0, os.path.join(REPO_DIR, 'RF_DETR_split_ver'))

# ── 저장소에서 그대로 재사용하는 함수 목록 ─────────────────────────────
from dataset import (
    load_raw_annotations,     # 박스당 1개로 흩어진 원본 annotation을 파일명 기준 병합 로드
    apply_corrections,        # corrections.json 기반 bbox 좌표/중복/누락/클래스 오기재 수정
    build_category_mapping,   # category_id 오름차순 -> 라벨 1~N 매핑
    make_folds,               # StratifiedGroupKFold 5-fold 분할
    cache_images,             # 이미지 로컬 캐시 (폴더 하나당 1회 호출)
    write_fold_dirs,          # fold별 {train,valid}에 COCO json + 이미지 배치
    save_label_map,           # label_map.json 저장 (run_kfold가 읽어감)
    compute_label_counts,     # 클래스(라벨)별 박스 수 집계
    find_input_dir,           # /kaggle/input 아래에서 폴더 "이름"으로 재귀 탐색 (find_data_root의 Kaggle용 짝)
)
from model import get_rfdetr_model          # RF-DETR variant 생성 (checkpoint_path로 가중치 로드)
from train import run_kfold                 # fold 학습 + fold별 리포팅(mAP/오답 시각화) 루프
from utils import (
    summarize_kfold_results,  # fold 평균 mAP 요약 출력
    summarize_per_class,      # 클래스별 mAP 집계 DataFrame
    show_error_gallery,       # 저장된 PNG 폴더를 grid로 보여주는 범용 뷰어
)
from visualize import (
    collect_predictions_ensemble,  # test 폴더에 대한 fold 모델 합집합 추론
    show_gt_class_crops,           # 클래스별 GT bbox crop grid 시각화 (7번 셀)
    show_pred_class_crops,         # 클래스별 예측 crop grid 시각화, score 표시 (15번 셀)
)
# ── RF_DETR_split_ver 공용 모듈 - 여러 노트북에서 반복되던 fold/앙상블 후처리 로직을 모듈화한 것 ──
#    (WBF/제출 CSV의 원리는 각 함수 docstring 참고 - 이전에는 이 노트북에 로컬 함수로 풀어져 있었음)
from folds import summarize_fold_distribution, print_fold_warnings  # fold별 이미지/박스 수 + 누락 클래스 점검
from ensemble import fuse_predictions_wbf, make_submission, extract_image_id  # WBF 융합 + 제출 CSV 생성

CORRECTIONS_PATH = os.path.join(REPO_DIR, 'RF_DETR_split_ver', 'corrections.json')
print('corrections.json 존재:', os.path.exists(CORRECTIONS_PATH))

In [ ]:
# [3. 입력 경로 자동 탐색]
#  /kaggle/input 아래에서 폴더 "이름"으로 찾기 때문에 competition/데이터셋 슬러그를 몰라도 동작합니다
#  (dataset.find_input_dir - 저장소 공용 함수). 탐색 실패 시 None이 출력되므로, 그 경우 아래 상수에
#  실제 경로를 직접 채워 넣으세요.
TRAIN_IMG_DIR = find_input_dir('train_images')       # 원본 train 이미지 (png)
TRAIN_ANN_DIR = find_input_dir('train_annotations')  # 원본 annotation (박스당 json 1개)
TEST_IMG_DIR  = find_input_dir('test_images')        # 제출 대상 test 이미지

paths = {'TRAIN_IMG_DIR': TRAIN_IMG_DIR, 'TRAIN_ANN_DIR': TRAIN_ANN_DIR, 'TEST_IMG_DIR': TEST_IMG_DIR}
for k, v in paths.items():
    print(f'{k:14s}:', v)
assert all(paths.values()), "자동 탐색에 실패한 경로가 있습니다. 위 상수에 직접 경로를 지정하세요."


In [ ]:
# [4. 실험 설정]
#  Task 0/1/2 비교 실험이므로 학습 하이퍼파라미터는 세 노트북 모두 아래와 동일하게 고정합니다.
#   - RF-DETR medium / 100 epochs / early stopping 사용 / cosine LR
#   - batch_size 4 x grad_accum_steps 4 = 유효 배치 16 (Kaggle 16GB GPU 기준, OOM 시 2x8로 조정)
#   - 저장소 config.yaml(Colab 경로 기준)은 수정하지 않고, 같은 "스키마"의 dict를 노트북에서 직접 정의합니다.
#     (train.run_kfold / train_fold가 이 스키마를 그대로 소비하므로 저장소 함수와 완전 호환)
TASK_ID = 0

config = {
    'data': {
        'dataset_dir': '/kaggle/tmp/dataset',    # fold 디렉토리 루트 (세션 임시 영역 - output 용량 절약)
        'cache_dir':   '/kaggle/tmp/img_cache',  # 이미지 로컬 캐시
        'n_splits': 5,
        'seed': 42,
    },
    'model': {
        'variant': 'medium',
        'tag': 'medium_task0_base',                        # 체크포인트/그래프/제출 파일명에 사용되는 실험 태그
    },
    'train': {
        'epochs': 100,
        'batch_size': 4,
        'grad_accum_steps': 4,
        'lr': 1e-4,
        'lr_encoder': 1.5e-4,
        'weight_decay': 1e-4,
        'lr_scheduler': 'cosine',
        'warmup_epochs': 0.0,
        'lr_min_factor': 0.0,
        'early_stopping': True,
        'early_stopping_patience': 10,
        'early_stopping_min_delta': 0.001,
        'tensorboard': False,
    },
    'output': {
        'local_output_dir': '/kaggle/tmp/outputs',   # fold 학습 중 임시 산출물
        'backup_dir': '/kaggle/working/outputs',     # best 체크포인트/학습곡선/오답 이미지 (커밋 시 보존됨)
    },
}

COLLECT_SCORE_THR = 0.05   # test 추론 "수집" threshold (WBF 전 단계이므로 낮게 잡아 recall 확보)
WBF_IOU_THR       = 0.55   # WBF에서 같은 객체로 간주할 IoU 기준
SUBMIT_SCORE_THR  = 0.3    # 제출 CSV에 포함할 최소 confidence (mAP 채점 가정)

for d in (config['data']['cache_dir'], config['data']['dataset_dir'],
          config['output']['local_output_dir'], config['output']['backup_dir']):
    os.makedirs(d, exist_ok=True)

SUBMISSION_PATH = f"/kaggle/working/submission_task{TASK_ID}_{config['model']['tag']}.csv"
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU(!) - 가속기 설정 확인')
print('제출 파일 경로:', SUBMISSION_PATH)

In [ ]:
# [5. 원본 train 로드 + annotation 수정]
#  원본은 "박스 1개당 JSON 1개"로 흩어져 있어 load_raw_annotations()가 파일명 기준으로 병합합니다.
#  이어서 corrections.json(팀에서 검수한 좌표 오류/중복/누락/클래스 오기재 내역)을 적용합니다.
#  (apply_corrections는 coord_fix -> remove -> modify -> add -> fix_category 순서를 내부에서 보장)
boxes_by_image, cats_by_image, img_meta, ids_by_image = load_raw_annotations(TRAIN_ANN_DIR)
print(f"수정 전: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")

boxes_by_image, cats_by_image = apply_corrections(
    boxes_by_image, cats_by_image, ids_by_image, CORRECTIONS_PATH)
print(f"수정 후: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")

# 원본 train의 클래스 목록(56종)을 보존해 둡니다.
#  - Task1: pool이 이 범위를 벗어나지 않는지 검증에 사용
#  - Task2: "train 56종 -> 라벨 1~56" 매핑 검증에 사용
train_cats = sorted({c for cs in cats_by_image.values() for c in cs})
print('train 클래스 수:', len(train_cats))

## 6. 데이터 병합 — Task 0은 생략

Baseline은 **원본 train만** 사용하므로 병합 단계가 없습니다.
(Task 1/2 노트북과 셀 번호를 맞추기 위해 자리만 남겨둡니다 — 이후 셀은 세 노트북이 동일한 흐름)

In [ ]:
# [7. 병합 데이터 시각화] 클래스별로 GT bbox를 잘라(grid) 모아봅니다.
#  GT bbox crop 시각화는 visualize.show_gt_class_crops (저장소에는 "예측" crop 유틸만 있어
#  GT용으로 별도 추가된 공용 함수 - ids_by_image가 없으면 빈 dict를 넘기면 됩니다).
IMG_PATH_INDEX = {os.path.basename(p): p
                  for p in glob.glob(os.path.join(TRAIN_IMG_DIR, '**', '*.png'), recursive=True)}
print('이미지 경로 인덱스:', len(IMG_PATH_INDEX), '개 파일')

# 전체 클래스를 표시합니다. 특정 클래스만 보려면 classes=[3351, 16548] 처럼 지정하세요.
show_gt_class_crops(boxes_by_image, cats_by_image, {}, IMG_PATH_INDEX, ncols=6)


In [ ]:
# [8. 카테고리 라벨 매핑] 원본 category_id -> 모델 라벨 (56종 -> 1~56)
#  build_category_mapping()은 category_id "오름차순"으로 1부터 라벨을 부여합니다.
#  합성 pool JSON의 카테고리 체계(1~56, name=원본 id)도 같은 오름차순 규칙으로 확인되었고,
#  Task2(74종)에서도 1~56 구간은 동일 -> 세 Task의 라벨 네임스페이스가 서로 호환됩니다.
#  (라벨 0은 RF-DETR가 요구하는 더미 배경 카테고리 'pill'로 비워둠 - build_coco가 자동 처리)
all_cats, cat2label, label2cat = build_category_mapping(cats_by_image)
assert len(all_cats) == 56, f'train 클래스 수가 56이 아님: {len(all_cats)} (데이터 구성 확인 필요)'
print('클래스 수:', len(all_cats), '| 라벨 범위: 1 ~', len(all_cats))

In [ ]:
# [9. 5-fold 분할 + fold별 클래스 배분 점검]
#  make_folds()는 StratifiedGroupKFold를 사용합니다 (저장소 함수 그대로).
#   - group: 파일명의 알약 구성 코드('_0_2' 앞부분) -> 같은 구성이 train/valid에 섞이지 않음
#     합성 이미지(syn_*.png)는 '_0_2' 패턴이 없어 이미지 1장 = 그룹 1개로 취급됩니다.
#   - 층화 기준: 이미지 내 "가장 희소한 클래스" -> 희소 클래스가 fold에 고르게 분산
#  fold별 분포 점검은 folds.summarize_fold_distribution (fold당 이미지/박스 수 + 빠진 라벨 확인).
file_names = sorted(boxes_by_image)
folds = make_folds(file_names, boxes_by_image, cats_by_image, cat2label,
                   config['data']['n_splits'], config['data']['seed'])

fold_summary, valid_pivot = summarize_fold_distribution(folds, file_names, cats_by_image, cat2label)
display(fold_summary)

# train에서 통째로 빠진 클래스가 있는 fold는 그 클래스를 전혀 학습하지 못하므로 경고로 표시
print_fold_warnings(fold_summary)

valid_pivot   # 라벨 x fold별 valid 박스 수 (셀 마지막 줄 -> 표로 표시)


In [ ]:
# [10. fold 디렉토리 생성]
#  이미지를 로컬 캐시에 1회 복사한 뒤, fold별 {train,valid} 폴더에 COCO json + 이미지를 배치합니다.
#  (전부 저장소 함수 - cache_images는 폴더 하나당 1회씩 호출하면 같은 캐시에 누적됩니다)
cache_images(TRAIN_IMG_DIR, config['data']['cache_dir'])

write_fold_dirs(folds, file_names, boxes_by_image, cats_by_image, img_meta,
                cat2label, all_cats, config['data']['cache_dir'], config['data']['dataset_dir'])

# run_kfold()가 학습 전에 이 label_map.json을 읽으므로 반드시 학습 전에 저장해야 합니다.
save_label_map(cat2label, label2cat, config['data']['dataset_dir'])
print('label_map.json 저장 완료')

## 11. 5-fold 학습

`run_kfold(config)` 하나로 fold마다 다음이 자동 수행됩니다 (모두 저장소 함수):

1. `train_fold()` — 학습 후 best 체크포인트를 `backup_dir`에 백업, **학습곡선 그래프 출력·저장**
2. `report_fold_result()` — valid셋 추론 1회로 **mAP 계산 + 오답 이미지 저장** (`{tag}_fold{i}_errors/`)

**이어하기**: `train_fold()`는 `backup_dir`에 해당 fold의 `*_best.pth`가 이미 있으면 학습을 건너뜁니다.
세션이 끊기면 이전 커밋의 output을 Input으로 추가하고 아래 선택 셀로 복사한 뒤 다시 실행하세요.

⏱ **시간 주의**: medium × 100 epochs × 5 folds는 12h 세션 한도를 넘길 수 있습니다.
처음에는 `run_kfold(config, max_folds=1)`로 fold 1개 소요 시간을 가늠해 보는 것을 권장합니다.

In [ ]:
# (선택) 이어하기: 이전 노트북 커밋의 output(outputs/*.pth)을 Input으로 추가했다면 backup_dir로 복사
#  -> train_fold()가 이미 학습된 fold를 자동으로 건너뜁니다.
# PREV_OUTPUT_DIR = '/kaggle/input/<이전-노트북-output-슬러그>/outputs'
# for p in glob.glob(os.path.join(PREV_OUTPUT_DIR, '*_best.pth')):
#     shutil.copy(p, config['output']['backup_dir'])
#     print('복사:', os.path.basename(p))

In [ ]:
# [11. 5-fold 학습 실행]
#  fold가 끝날 때마다 학습곡선 그래프와 valid mAP, 오답 이미지 저장 경로가 출력됩니다.
#  소요 시간 가늠용 sanity check: run_result = run_kfold(config, max_folds=1)
run_result = run_kfold(config)

In [ ]:
# [12. 결과 요약]
#  - summarize_kfold_results: fold 평균 mAP (mAP@0.75:0.95 평균±표준편차)
#  - summarize_per_class: 클래스별 mAP를 전 fold에 걸쳐 집계 (데이터 내 등장 횟수 포함)
kfold_summary = summarize_kfold_results(run_result['fold_metrics'], config['model']['tag'])

label_counts = compute_label_counts(config['data']['dataset_dir'])
per_class_df = summarize_per_class(run_result['fold_metrics'],
                                   run_result['label_to_category_id'], label_counts)
per_class_df   # mean_AP 내림차순 - 하위 클래스가 보강 대상/효과 확인 포인트

In [ ]:
# [12-1. 오답 이미지 확인] run_kfold가 fold별로 저장해둔 오답 이미지를 grid로 봅니다.
#  fold 번호/페이지(start, limit)를 바꿔가며 확인하세요. (GT=초록, 예측=빨강, 파일명에 순번 포함)
show_error_gallery(
    os.path.join(config['output']['backup_dir'], f"{config['model']['tag']}_fold0_errors"),
    ncols=3, limit=6,
)

## 13~16. test 추론 → WBF 앙상블 → 클래스별 시각화 → 제출

test에는 GT가 없으므로 mAP 계산 없이:
5개 fold 모델의 예측을 전부 수집(합집합, 저장소 `collect_predictions_ensemble`) → **WBF**로 이미지 단위 융합 →
클래스별 crop 시각화(confidence 표시)로 육안 점검 → 제출 CSV 생성 순서로 진행합니다.

In [ ]:
# [13. test 추론 - 5 fold 합집합 수집]
#  각 fold의 best 체크포인트로 모델 5개를 만들어, test 이미지마다 5개 모델의 예측을 전부 모읍니다.
#  WBF 융합 전이므로 낮은 threshold(COLLECT_SCORE_THR)로 수집하고, 제출 단계에서 최종 필터링합니다.
ckpts = [p for p in run_result['checkpoints'] if p]
assert len(ckpts) == config['data']['n_splits'], f"체크포인트 누락: {run_result['checkpoints']}"

models = [get_rfdetr_model(config['model']['variant'], checkpoint_path=p) for p in ckpts]
test_pred_data = collect_predictions_ensemble(models, TEST_IMG_DIR,
                                              score_threshold=COLLECT_SCORE_THR)
print('test 이미지 수:', len(test_pred_data))

# RF-DETR 내부 예약 라벨(배경 0 등, label_map에 없는 라벨) 제거
#  - 저장소 report_fold_result()가 valid 평가 때 하는 것과 동일한 정제 과정입니다.
valid_labels = set(label2cat)
for d in test_pred_data:
    keep = torch.tensor([int(l) in valid_labels for l in d['pred_labels']], dtype=torch.bool)
    for k in ('pred_boxes', 'pred_labels', 'pred_scores', 'pred_fold'):
        d[k] = d[k][keep]
print('합집합 예측 박스 수:', sum(len(d['pred_boxes']) for d in test_pred_data))

In [ ]:
# [14. WBF(Weighted Box Fusion) 융합]
#  합집합 그대로 제출하면 같은 알약이 fold 수만큼 중복되어 mAP에서 FP로 깎입니다.
#  저장소의 _cluster_same_class_boxes(대표 박스 1개 선택)와 달리, WBF는 겹치는 박스들의 좌표를
#  confidence 가중 평균해 더 정교한 박스를 만듭니다 (ensemble.fuse_predictions_wbf, 외부 패키지
#  ensemble-boxes 사용). conf_type='avg'(기본값) 기준, 일부 fold만 잡은 박스는 score가
#  (동의 모델 수/전체 모델 수) 비율로 낮아져 "몇 개 fold가 동의했는지" 신호가 score에 자연스럽게 반영됩니다.
fused_data = fuse_predictions_wbf(test_pred_data, n_models=len(models),
                                  iou_thr=WBF_IOU_THR, skip_box_thr=COLLECT_SCORE_THR)
n_before = sum(len(d['pred_boxes']) for d in test_pred_data)
n_after = sum(len(d['pred_boxes']) for d in fused_data)
print(f'융합 전 박스 {n_before}개 -> 융합 후 {n_after}개')


In [ ]:
# [15. 추론 결과 클래스별 시각화] WBF 융합 예측을 예측 클래스별 crop grid로 표시합니다.
#  각 crop 제목에 confidence score + 파일명이 표시됩니다 (visualize.show_pred_class_crops).
#  제출 기준(SUBMIT_SCORE_THR)과 동일한 threshold로 점검하는 것을 권장합니다.
show_pred_class_crops(fused_data, label2cat, score_thr=SUBMIT_SCORE_THR, max_per_class=8)


In [ ]:
# [16. 제출 CSV 생성]
#  요구 포맷: annotation_id, image_id, category_id, bbox_x, bbox_y, bbox_w, bbox_h, score
#   - image_id: 이미지 "파일명의 숫자" (ensemble.extract_image_id, 규칙이 다르면 그 함수를 고쳐야 함)
#   - category_id: 원본 category_id (모델 라벨 -> label2cat 역매핑)
#   - bbox: xyxy(내부 표현) -> xywh(COCO)로 변환 (ensemble.make_submission)
# 파일명 -> image_id 매핑을 눈으로 먼저 확인 (규칙이 다르면 extract_image_id 수정)
for d in fused_data[:5]:
    print(d['file_name'], '->', extract_image_id(d['file_name']))

submission_df = make_submission(fused_data, label2cat, SUBMIT_SCORE_THR, SUBMISSION_PATH)
submission_df.head(10)
